# [실습] 전처리 파이프라인과 데이터 누수 — 점수를 믿어도 되는가

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# Chapter 4 실습. 이 점수를 믿어도 되는가?

**필수 머신러닝 · Chapter 4**

| Part | 주제 |
| --- | --- |
| Part 1 | 너무 좋은 점수 의심하기 |
| Part 2 | 전처리를 올바른 순서로 |
| Part 3 | 점수를 믿는 법: 교차검증과 시간 순 분할 |
| Part 4 | 보고할 숫자 정하기 |

## 분석 의뢰서

> **의뢰 부서**: 은행 마케팅팀 (Chapter 3의 후속 의뢰)
> **목적**: 지난번 모델을 다음 캠페인에 실제로 쓰려 한다. 그런데 팀원 한 명이 통화 시간 변수를 추가했더니 점수가 훨씬 좋아졌다고 한다.
> **제약**: 모델이 약속한 가입 수가 실제와 크게 다르면 캠페인 예산 승인을 받을 수 없다.
> **질문**: 어느 모델의 점수를 믿고 보고해야 하는가? 그 점수가 실제 캠페인에서도 나온다고 어떻게 확신하는가?

## 이 실습의 질문

> **통화 시간을 뺀 모델과 넣은 모델, 무작위 분할과 시간 순 분할 — 마케팅팀에 "다음 캠페인에서 기대할 가입 수"로 보고할 숫자는 어느 것인가? 그리고 그 숫자를 믿을 수 있다고 어떻게 설득할 것인가?**

이 실습에는 새 모델이 나오지 않습니다. Chapter 3의 모델을 그대로 쓰면서, 그 점수를 **의심하고, 고쳐 만들고, 다시 재는** 연습을 합니다.

## 이 노트북을 쓰는 법

| 표시 | 뜻 |
| --- | --- |
| **[직접 작성]** | 빈 코드 셀을 직접 채웁니다. 강의 교안의 코드를 보지 않고 먼저 써 보세요 |
| **[판단]** | 코드가 아니라 생각을 적는 칸입니다. 셀을 더블클릭해 `나의 답:` 뒤에 적으세요 |
| **[제공]** | 미리 채워 둔 코드입니다. 읽고 실행하세요 |
| **힌트** | 막혔을 때만 펼치세요 |
| **먼저 예상하기** | 결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다 |

## 준비

Chapter 3과 같은 `bank-full.csv`(통화 45,211건)를 이 노트북과 같은 폴더에 두세요. 이번에는 Chapter 3에서 뺐던 통화 시간(`duration`) 열을 **지우지 않고** 둡니다.

In [2]:
# [제공] 이 실습에서 쓰는 라이브러리
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# 그래프의 한글 글꼴 (환경에 맞는 글꼴이 없으면 글자가 깨질 수 있습니다)
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 은행 마케팅 데이터프레임 불러오기
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 
  
# data (as pandas dataframes) 
X = bank_marketing.data.features 
y = bank_marketing.data.targets 
  
# metadata 
print(bank_marketing.metadata) 
  
# variable information 
print(bank_marketing.variables) 

df = pd.concat([bank_marketing.data.features, bank_marketing.data.targets], axis=1)
print(df.shape)
df.head()

print(df.isna().sum())            # ucimlrepo 데이터: NaN 개수
print((df == 'unknown').sum())    # 0이면 NaN 방식

---
# Part 1. 너무 좋은 점수 의심하기

## 1-1. 준비: Chapter 3의 데이터 다시 만들기

**[제공]** Chapter 3에서 한 준비를 한 번에 합니다. 데이터를 불러와 `target`을 만들고, 같은 방법(80 : 20, `random_state=42`, `stratify`)으로 나눕니다. 이번에는 `duration` 열을 지우지 않습니다.

| 이름 | 뜻 |
| --- | --- |
| `num_cols` | 숫자 피처 4개: `age`, `balance`, `campaign`, `previous` |
| `cat_cols` | 범주 피처 8개: `poutcome`, `housing`, `loan`, `contact`, `month`, `job`, `marital`, `education` |
| `features` | 두 목록을 합친 Chapter 3 모델의 피처 12개 |

In [5]:
# 데이터 준비 (Chapter 3과 같음, duration은 남겨 둠)
df["target"] = (df["y"] == "yes").astype(int)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["target"])

summary = pd.DataFrame({
    "의미": ["미가입", "가입"],
    "개수": counts.values,
    "비율(%)": (counts / counts.sum() * 100).round(2).values,
}, index=pd.Index(counts.index, name="target"))

summary.style.format({"개수": "{:,}", "비율(%)": "{:.1f}%"})

num_cols = ["age", "balance", "campaign", "previous"]
cat_cols = ["poutcome", "housing", "loan", "contact", "month", "job", "marital", "education"]
features = num_cols + cat_cols
y_train, y_test = train_df["target"], test_df["target"]
print(len(train_df), len(test_df), "| 평가용 가입률", round(y_test.mean(), 3))

36168 9043 | 평가용 가입률 0.117


### 실습 도구 (실행만 하세요)

| 도구 | 하는 일 |
| --- | --- |
| `record(이름, 정답, 확률, 분할)` | 점수를 결과표에 남긴다: ROC AUC, PR AUC, 평가 데이터의 가입률, 상위 1,000명 중 가입 수, 무작위로 1,000명을 골랐을 때의 기대 가입 수, 배수 |
| `show_log()` | 결과표를 보여 준다 |

In [6]:
# [제공] 실습 도구
results = []

def record(name, y_true, prob, split="무작위"):
    y_true = np.asarray(y_true); prob = np.asarray(prob)
    top = y_true[np.argsort(-prob, kind="stable")][:1000].sum()
    base = y_true.mean()
    row = {"실험": name, "분할": split,
           "ROC AUC": round(roc_auc_score(y_true, prob), 3),
           "PR AUC": round(average_precision_score(y_true, prob), 3),
           "평가 가입률": round(base, 3),
           "상위 1,000명 가입": int(top),
           "무작위 1,000명 기대": round(1000 * base),
           "배수": round(top / (1000 * base), 1)}
    results.append(row)
    print(f"기록: [{name} / {split}] ROC AUC {row['ROC AUC']:.3f} | 상위 1,000명 가입 {row['상위 1,000명 가입']}명 (무작위의 {row['배수']}배)")

def show_log():
    return pd.DataFrame(results)

## 1-2. Chapter 3의 모델 다시 만들기

**[직접 작성]** Chapter 3과 같은 방법으로 로지스틱 회귀를 만들고, 평가용 고객의 가입 확률을 `prob_base`에 저장한 뒤 `record("Chapter 3 모델", y_test, prob_base)`로 기록하세요.

1. `pd.get_dummies(df[features])`로 원핫 인코딩 → 학습용·평가용 행으로 나누기 (`.loc[train_df.index]`)
2. `StandardScaler`를 **학습용으로** `fit`하고, 학습용·평가용을 `transform`
3. `LogisticRegression(max_iter=3000)`으로 학습하고 `predict_proba(...)[:, 1]`

<details><summary>힌트</summary>

```python
X = pd.get_dummies(df[features]).astype(float)
X_tr, X_te = X.loc[train_df.index], X.loc[test_df.index]
scaler = StandardScaler().fit(X_tr)
model = LogisticRegression(max_iter=3000).fit(scaler.transform(X_tr), y_train)
prob_base = model.predict_proba(scaler.transform(X_te))[:, 1]
```
</details>

In [8]:
# 1. 원핫 인코딩 → 학습용·평가용 행으로 나누기
X = pd.get_dummies(df[features]).astype(float)
X_tr, X_te = X.loc[train_df.index], X.loc[test_df.index]

# 2. 스케일러 기준은 학습용으로만 잡고, 양쪽에 적용
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

# 3. 로지스틱 회귀 학습 → 평가용 가입 확률
model = LogisticRegression(max_iter=3000).fit(X_tr_s, y_train)
prob_base = model.predict_proba(X_te_s)[:, 1]

record("Chapter 3 모델", y_test, prob_base)

기록: [Chapter 3 모델 / 무작위] ROC AUC 0.773 | 상위 1,000명 가입 464명 (무작위의 4.0배)


결과가 Chapter 3과 같은지(ROC AUC 0.773 안팎) 확인하세요.

## 1-3. 통화 시간을 넣어 보면

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. Chapter 3 실습에서 피처를 더하고 설정을 바꿨을 때 ROC AUC가 얼마나 움직였는지 떠올려 보세요.

> **[판단]** 예상: 통화 시간(`duration`, 초)을 피처에 하나 더하면 ROC AUC가 0.773에서 얼마가 될까요?
>
> 나의 답: 0.80 안팎까지 오를 것 같다. Chapter 3에서 피처나 설정을 바꿨을 때 ROC AUC는 소수점 둘째 자리 정도만 움직였으니, 변수 하나로는 +0.02~0.03이면 큰 변화라고 예상한다.

**[직접 작성]** 1-2의 코드에서 피처 목록을 `features + ["duration"]`으로 바꿔 같은 모델을 만들고, 확률을 `prob_dur`에 저장한 뒤 `record("통화 시간 추가", y_test, prob_dur)`로 기록하세요.

In [9]:
# 1. 피처에 duration을 더해 원핫 인코딩 → 학습용·평가용 행으로 나누기
features_dur = features + ["duration"]
X_dur = pd.get_dummies(df[features_dur]).astype(float)
X_tr_d, X_te_d = X_dur.loc[train_df.index], X_dur.loc[test_df.index]

# 2. 스케일러 기준은 학습용으로만
scaler_d = StandardScaler().fit(X_tr_d)
X_tr_ds, X_te_ds = scaler_d.transform(X_tr_d), scaler_d.transform(X_te_d)

# 3. 같은 로지스틱 회귀 → 평가용 가입 확률
model_dur = LogisticRegression(max_iter=3000).fit(X_tr_ds, y_train)
prob_dur = model_dur.predict_proba(X_te_ds)[:, 1]

record("통화 시간 추가", y_test, prob_dur)


기록: [통화 시간 추가 / 무작위] ROC AUC 0.906 | 상위 1,000명 가입 579명 (무작위의 4.9배)


> **[판단]** ROC AUC가 얼마나 올랐나요? 예상과 비교해 보세요. Chapter 3 실습에서 다른 피처를 더했을 때의 변화와 견주면, 이 변화는 어느 정도로 큰가요? 이 결과를 보고 먼저 기뻐해야 할까요, 먼저 의심해야 할까요?
>
> 나의 답: 0.773 → 0.906으로 +0.133 올랐다. 상위 1,000명 가입도 464명 → 579명(무작위의 4.0배 → 4.9배). 예상(+0.02~0.03)보다 다섯 배 넘게 컸다. 이 실습의 다른 비교에서 설정을 바꿨을 때 차이가 0.002~0.06 수준인 것과 견주면 변수 하나로 생긴 변화치고 비정상적으로 크다. 먼저 의심해야 한다. 변수 하나가 점수를 이만큼 끌어올리면 정답 정보가 섞여 들어온 것은 아닌지부터 확인해야 한다.

## 1-4. 이 변수는 어디서 오는가

**[직접 작성]** **학습용 데이터**에서 다음 두 가지를 확인하세요.

1. 가입(`target` = 1)과 미가입(0) 고객의 통화 시간 하위 25%, 중앙값, 상위 25%
2. 통화 시간이 60초 미만인 통화와 600초(10분) 초과인 통화의 가입률과 통화 수

<details><summary>힌트</summary>

1번은 `train_df.groupby("target")["duration"].quantile([0.25, 0.5, 0.75]).unstack()`.
2번은 조건을 만들어 `train_df.loc[조건, "target"].agg(["mean", "size"])`.
</details>

In [10]:
# 1. 가입/미가입별 통화 시간 분위수
print(train_df.groupby("target")["duration"].quantile([0.25, 0.5, 0.75]).unstack())
print()

# 2. 짧은 통화 vs 긴 통화의 가입률과 통화 수
short = train_df["duration"] < 60
long_ = train_df["duration"] > 600
print("60초 미만:\n", train_df.loc[short, "target"].agg(["mean", "size"]))
print("600초 초과:\n", train_df.loc[long_, "target"].agg(["mean", "size"]))


         0.25   0.50   0.75
target                     
0        95.0  164.0  278.0
1       245.0  426.0  734.5

60초 미만:
 mean       0.00161
size    3727.00000
Name: target, dtype: float64
600초 초과:
 mean       0.486034
size    3043.000000
Name: target, dtype: float64


> **[판단]** 통화 시간과 가입 사이의 관계는 어느 방향일까요? 통화가 길어서 가입한 것일까요, 가입할 고객이라서 통화가 길어진 것일까요? 마케팅팀이 "누구에게 전화할까"를 정하는 시점에 이 값을 알 수 있나요?
>
> 나의 답: 가입할 고객이라서 통화가 길어진 쪽이다. 학습용에서 가입 고객의 통화 시간 중앙값은 426초, 미가입 고객은 164초다. 60초 미만 통화 3,727건의 가입률은 0.16%(사실상 0)이고, 600초 초과 3,043건은 48.6%로 전체 가입률(11.7%)의 4배가 넘는다. 관심 없는 고객은 금방 끊고, 가입하는 고객은 상담·절차 때문에 통화가 길어진다. 통화 시간은 가입 결정의 결과에 가깝다. 게다가 통화가 끝나야 기록되는 값이라, "누구에게 전화할까"를 정하는 시점에는 알 수 없다.

## 1-5. 피처마다 예측 시점 점검하기

데이터 설명서에 적힌 각 열의 뜻입니다. 마케팅팀은 **전화를 걸기 전에** 누구에게 걸지 정해야 합니다.

| 열 | 설명서의 뜻 |
| --- | --- |
| `age`, `job`, `marital`, `education` | 고객 정보 |
| `balance`, `housing`, `loan` | 계좌 잔액, 주택 대출, 개인 대출 |
| `contact` | 연락 수단 |
| `month`, `day` | 마지막 통화의 월, 일 |
| `campaign` | 이번 캠페인에서 이 고객에게 건 통화 수 (**마지막 통화 포함**) |
| `pdays`, `previous`, `poutcome` | 이전 캠페인 이후 경과일, 이전 캠페인 통화 수, 이전 캠페인 결과 |
| `duration` | 마지막 통화의 시간 (초) |

> **[판단]** 위 열 중 예측 시점(전화를 걸기 전)에 **알 수 없는** 열과 **애매한** 열을 고르고, 각각 이유를 적으세요. (힌트: `campaign`의 "마지막 통화 포함"을 눈여겨보세요.)
>
> 나의 답: 알 수 없는 열: duration. 마지막 통화가 끝나야 기록되므로 전화를 걸기 전에는 존재하지 않는다. 애매한 열: campaign은 '마지막 통화 포함'이라 이번 캠페인이 끝난 뒤에야 확정된다. 가입한 고객은 그 통화에서 멈추고, 미가입 고객은 계속 전화를 받으므로 결과가 섞여 있다. month·day도 '마지막 통화'의 날짜라 언제 연락이 끝났는지에 결과가 반영될 수 있다. 다만 '오늘 전화한다면'으로 바꿔 쓰면 쓸 수 있다. 쓸 수 있는 열: age, job, marital, education, balance, housing, loan, contact는 전화 전에 아는 고객 정보다. pdays, previous, poutcome도 '이전' 캠페인 기록이라 전화 전에 안다.

---
# Part 2. 전처리를 올바른 순서로

Part 1의 누수는 **변수**에 숨어 있었습니다. 이 Part의 누수는 **코드의 순서**에 숨어 있습니다. 이제부터는 통화 시간을 쓰지 않고, Chapter 3 모델의 피처 12개(`features`)만 씁니다.

## 2-1. 스케일러의 기준을 전체 데이터로 잡으면

**[직접 작성]** 1-2와 같은 모델을 만들되, 스케일러를 **전체 데이터(학습용 + 평가용)**로 `fit`한 버전의 확률을 `prob_leak`에 저장하세요. 그다음 `prob_base`와 `prob_leak`의 ROC AUC를 소수점 넷째 자리까지 출력해 비교하세요. (이 실험은 결과표에 기록하지 않습니다.)

<details><summary>힌트</summary>

`StandardScaler().fit(X)`처럼 나누기 전의 `X` 전체로 `fit`하면 됩니다. 나머지는 1-2와 같습니다. 출력은 `print(round(roc_auc_score(y_test, prob_leak), 4))`.
</details>

In [11]:
# 1-2의 X(전체 데이터 원핫)를 그대로 사용 — 스케일러 기준만 전체로
scaler_leak = StandardScaler().fit(X)
model_leak = LogisticRegression(max_iter=3000).fit(scaler_leak.transform(X_tr), y_train)
prob_leak = model_leak.predict_proba(scaler_leak.transform(X_te))[:, 1]

print("올바른 방법:", round(roc_auc_score(y_test, prob_base), 4))
print("누수 방법  :", round(roc_auc_score(y_test, prob_leak), 4))


올바른 방법: 0.7728
누수 방법  : 0.7728


> **[판단]** 두 점수는 얼마나 다른가요? 왜 이런 결과가 나왔을까요? 점수에 차이가 없다면, 그래도 "기준은 학습용에서만" 규칙을 지켜야 하는 이유는 무엇일까요?
>
> 나의 답: 0.7728 vs 0.7728로 소수점 넷째 자리까지 같다. 표준화는 평균을 빼고 표준편차로 나누는 선형 변환일 뿐이고, 학습용(36,168건)과 전체(45,211건)의 평균·표준편차가 거의 같다. 그래서 계수만 조금 바뀌고 고객 순서(ROC AUC)는 그대로다. 정답(target) 정보도 들어가지 않았다. 그래도 규칙을 지켜야 하는 이유: ① 운영에서는 평가용(미래 고객)이 학습 시점에 없으니, 전체로 fit하는 코드는 실제로 재현할 수 없는 점수를 만든다. ② 데이터가 작거나 분포가 시간에 따라 변하면, 또는 결측값 채우기·타깃 인코딩·피처 선택처럼 정보를 더 많이 쓰는 전처리에서는 차이가 커진다(2-2). ③ 차이가 없을 때도 규칙을 지키는 습관이 있어야, 차이가 날 때 실수하지 않는다.

## 2-2. 기준에 타깃이 들어가면: 타깃 인코딩

**타깃 인코딩**은 범주를 "그 범주의 가입률"로 바꾸는 인코딩입니다. 여기서는 통화한 날(월 + 일, 예: `may-12`)을 범주로 써서, 날마다 가입률을 계산해 피처 하나로 만듭니다.

**[제공]** 날짜 범주를 만듭니다.

In [13]:
# [제공] 통화한 날(월-일) 범주
day_key = df["month"] + "-" + df["day_of_week"].astype(str)
key_tr, key_te = day_key.loc[train_df.index], day_key.loc[test_df.index]
print("날짜 범주 수:", day_key.nunique(), "| 날짜당 통화 수 중앙값:", int(day_key.value_counts().median()))

날짜 범주 수: 318 | 날짜당 통화 수 중앙값: 29


**[직접 작성]** 날짜별 가입률을 두 가지 방법으로 계산해, 각각 **피처 하나짜리** 로지스틱 회귀를 만들고 평가용 ROC AUC를 비교하세요.

- **올바른 방법**: 가입률을 **학습용 데이터로만** 계산한다 (`train_df["target"].groupby(key_tr).mean()`)
- **누수가 있는 방법**: 가입률을 **전체 데이터로** 계산한다 (`df["target"].groupby(day_key).mean()`)

어느 방법이든 계산한 가입률을 학습용과 평가용 날짜에 `map`으로 붙여 피처로 씁니다. 학습용에 없던 날짜는 학습용 전체 가입률로 채웁니다.

<details><summary>힌트</summary>

```python
rate = train_df["target"].groupby(key_tr).mean()
f_tr = key_tr.map(rate).fillna(y_train.mean()).to_frame()
f_te = key_te.map(rate).fillna(y_train.mean()).to_frame()
m = LogisticRegression().fit(f_tr, y_train)
print(roc_auc_score(y_test, m.predict_proba(f_te)[:, 1]))
```
누수가 있는 방법은 `rate`만 전체 데이터로 바꿉니다.
</details>

In [14]:
def target_enc_auc(rate):
    f_tr = key_tr.map(rate).fillna(y_train.mean()).to_frame()
    f_te = key_te.map(rate).fillna(y_train.mean()).to_frame()
    m = LogisticRegression().fit(f_tr, y_train)
    return roc_auc_score(y_test, m.predict_proba(f_te)[:, 1])

rate_ok   = train_df["target"].groupby(key_tr).mean()   # 학습용으로만
rate_leak = df["target"].groupby(day_key).mean()        # 전체 데이터로

print("올바른 방법:", round(target_enc_auc(rate_ok), 4))
print("누수 방법  :", round(target_enc_auc(rate_leak), 4))


올바른 방법: 0.7287
누수 방법  : 0.7481


> **[판단]** 이번에는 두 방법의 점수가 다른가요? 2-1의 스케일러와 무엇이 다르기에 이번에는 차이가 났을까요? "그날 통화가 몇 건뿐인 날"이라면 누수가 더 커질까요, 작아질까요?
>
> 나의 답: 다르다. 올바른 방법 0.7287 vs 누수 방법 0.7481로, 누수 쪽이 0.019 높다. 스케일러는 피처의 평균·표준편차만 썼지만, 타깃 인코딩은 기준 자체가 정답(target)의 평균이다. 전체 데이터로 가입률을 계산하면 평가용 고객의 정답이 그대로 피처 값에 섞여 들어가, 모델이 시험 답을 미리 본 셈이 된다. 그날 통화가 몇 건뿐이면 누수가 더 커진다. 날짜 범주는 318개이고 날짜당 통화 수 중앙값은 29건이다. 통화가 적은 날일수록 평가용 한 명의 정답이 그날 가입률에서 차지하는 비중이 커서, 피처가 사실상 정답을 가리키게 된다.

## 2-3. `get_dummies`의 함정

운영에서는 하루치 고객이 조금씩 들어옵니다. **[제공]** 셀은 평가용 고객 중 20명만 뽑아 `get_dummies`로 바꾸고, 학습할 때의 열과 비교합니다.

In [15]:
# [제공] 새 고객 20명이 들어왔다면
new_customers = test_df.sample(20, random_state=1)
train_cols = pd.get_dummies(df[features]).columns
new_cols = pd.get_dummies(new_customers[features]).columns
print("학습할 때 열 수:", len(train_cols), "| 새 고객 20명의 열 수:", len(new_cols))
print("새 고객에게 없는 열 (예시 10개):", sorted(set(train_cols) - set(new_cols))[:10])

학습할 때 열 수: 42 | 새 고객 20명의 열 수: 35
새 고객에게 없는 열 (예시 10개): ['job_entrepreneur', 'job_self-employed', 'job_student', 'month_aug', 'month_jan', 'month_sep', 'poutcome_success']


> **[판단]** 열 수가 다르면 학습한 모델에 새 고객을 넣을 때 어떤 일이 생길까요? 지금까지 Chapter 1~3에서 이 문제를 겪지 않은 이유는 무엇일까요?
>
> 나의 답: 열 수가 다르면(학습 42개 vs 새 고객 20명 35개) 모델에 넣을 때 '피처 개수가 다르다'는 오류가 난다. 열 이름을 맞추지 않고 배열로만 넣으면 더 나쁘다. 오류 없이 엉뚱한 열에 엉뚱한 계수가 곱해져 틀린 확률이 조용히 나온다. Chapter 1~3에서 이 문제를 겪지 않은 이유는 항상 나누기 전의 df 전체에 get_dummies를 적용했기 때문이다. 평가용에만 있는 범주까지 미리 알고 열을 만든 셈이고, 이것도 평가용 정보를 빌려 쓴 것이다. 운영에서는 하루치 고객이 조금씩 들어오므로 이 방법을 쓸 수 없다.

## 2-4. 파이프라인으로 다시 만들기

**[직접 작성]** `ColumnTransformer`와 `Pipeline`으로 Chapter 3 모델을 다시 만드세요.

1. `preprocess`: 숫자 열(`num_cols`)에는 `StandardScaler()`, 범주 열(`cat_cols`)에는 `OneHotEncoder(handle_unknown="ignore")`
2. `pipe_model`: `Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])`
3. **학습용 데이터 그대로**(`train_df[features]`) `fit`하고, 평가용 확률을 `prob_pipe`에 저장해 `record("파이프라인", y_test, prob_pipe)`로 기록
4. 2-3의 `new_customers`에도 `predict_proba`가 동작하는지 확인

<details><summary>힌트</summary>

```python
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])
pipe_model.fit(train_df[features], y_train)
```
</details>

In [16]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])

pipe_model.fit(train_df[features], y_train)
prob_pipe = pipe_model.predict_proba(test_df[features])[:, 1]
record("파이프라인", y_test, prob_pipe)

# 새 고객 20명에도 그대로 동작하는지
print(pipe_model.predict_proba(new_customers[features])[:, 1].round(3))


기록: [파이프라인 / 무작위] ROC AUC 0.773 | 상위 1,000명 가입 465명 (무작위의 4.0배)
[0.22  0.18  0.117 0.076 0.578 0.075 0.098 0.04  0.094 0.136 0.017 0.054
 0.075 0.024 0.079 0.405 0.057 0.249 0.02  0.09 ]


> **[판단]** 파이프라인의 점수는 1-2의 "손으로 순서 맞추기"와 같은가요? 점수가 같다면, 파이프라인으로 바꿔서 얻은 것은 무엇인가요? 2-1, 2-3과 연결해 적어 보세요.
>
> 나의 답: 같다. ROC AUC 0.773으로 1-2와 같고, 상위 1,000명 가입도 464 vs 465로 사실상 같다. 점수는 그대로지만 얻은 것은 세 가지다. ① 2-1과 연결: 스케일러·인코더 기준이 fit할 때 넣은 학습용 데이터로만 자동으로 잡혀서, 순서를 틀릴 여지가 없다. ② 2-3과 연결: OneHotEncoder가 학습 때의 범주 목록을 기억하고(handle_unknown='ignore'), 새 고객 20명이 들어와도 같은 열 구조로 바꿔 예측한다(실제로 predict_proba가 동작했다). ③ 전처리 + 모델이 한 덩어리라 교차검증(3-1)과 운영 배포에 그대로 쓸 수 있다.

---
# Part 3. 점수를 믿는 법: 교차검증과 시간 순 분할

## 3-1. 5겹 교차검증

Chapter 1~3에서는 평가용 점수를 보며 설정(깊이, $\alpha$, 임계값)을 골랐습니다. 이제는 설정을 고르는 데 **교차검증**을 쓰고, 평가용은 마지막 확인에만 씁니다.

**[직접 작성]** 2-4의 `pipe_model`을 **학습용 데이터로** 5겹 교차검증하세요. 점수 다섯 개와 평균, 표준편차를 출력합니다.

<details><summary>힌트</summary>

```python
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe_model, train_df[features], y_train, cv=cv, scoring="roc_auc")
```
</details>

In [17]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe_model, train_df[features], y_train, cv=cv, scoring="roc_auc")

print("fold별:", scores.round(3))
print(f"평균 {scores.mean():.3f} ± {scores.std():.3f}")


fold별: [0.765 0.765 0.759 0.769 0.762]
평균 0.764 ± 0.003


> **[판단]** 교차검증 평균은 평가용 ROC AUC(0.773)와 얼마나 가깝나요? 교차검증에는 평가용 데이터가 쓰였나요? 파이프라인을 통째로 넣었다는 점은 왜 중요할까요?
>
> 나의 답: 교차검증 평균은 0.764 ± 0.003이고, 평가용 0.773보다 0.009 낮다. 표준편차의 3배쯤이라 아주 같지는 않지만 비슷한 수준이다. 평가용이 이번 분할에서 조금 운 좋게 나온 것으로 보인다. 교차검증에는 평가용 데이터가 쓰이지 않았다(train_df만 사용). 파이프라인을 통째로 넣은 것이 중요하다. 그래야 fold마다 스케일러·인코더가 그 fold의 학습 부분으로만 다시 fit되기 때문이다. 미리 전처리한 데이터를 넣으면 검증 fold의 정보가 전처리 기준에 섞여(2-1, 2-2의 누수) 교차검증 점수가 부풀 수 있다.

## 3-2. 두 모델을 비교할 때: 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 세 쌍의 모델을 같은 5겹 교차검증으로 비교합니다.

| 비교 | 모델 A | 모델 B |
| --- | --- | --- |
| ① | 로지스틱 회귀 ($C = 1$, 기본값) | 로지스틱 회귀 ($C = 0.01$, 규제를 강하게) |
| ② | 결정트리 깊이 5 | 결정트리 깊이 8 |
| ③ | 로지스틱 회귀 ($C = 1$) | 결정트리 깊이 8 |

> **[판단]** 예상: 세 비교 중 "B가 A보다 낫다(또는 못하다)"고 자신 있게 보고할 수 있는 비교는 어느 것일까요? 이유도 적어 보세요.
>
> 나의 답: ③(로지스틱 C=1 vs 트리 깊이 8)만 자신 있게 보고할 수 있을 것 같다. 모델 종류가 달라 점수 차이가 클 것이고, 앞 장에서도 트리 하나는 로지스틱보다 확률 순위가 거칠었다. ①은 C만 바꾼 것이라 차이가 아주 작을 것이고, ②는 깊이만 조금 바꾼 것이라 fold마다 이기는 쪽이 바뀔 수 있다.

## 3-3. fold별 차이로 비교하기

**[제공]** 모델을 쉽게 만드는 함수 `make_pipe`와, 두 모델을 **같은 5겹**으로 평가해 fold별 점수와 차이를 보여 주는 함수 `compare`입니다.

In [18]:
# [제공] 모델 만들기와 대응 비교
def make_pipe(clf, scale=True):
    pre = ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("pre", pre), ("clf", clf)])

def compare(name_a, model_a, name_b, model_b):
    s_a = cross_val_score(model_a, train_df[features], y_train, cv=cv, scoring="roc_auc")
    s_b = cross_val_score(model_b, train_df[features], y_train, cv=cv, scoring="roc_auc")
    diff = s_b - s_a
    print(f"{name_a}: {s_a.round(3)} | 평균 {s_a.mean():.3f} ± {s_a.std():.3f}")
    print(f"{name_b}: {s_b.round(3)} | 평균 {s_b.mean():.3f} ± {s_b.std():.3f}")
    print(f"fold별 차이 (B - A): {diff.round(3)} | 평균 차이 {diff.mean():+.3f}")
    print()

**[직접 작성]** `make_pipe`로 모델 네 개를 만들고, `compare`로 3-2의 세 쌍을 비교하세요. 결정트리는 스케일링이 필요 없으므로 `scale=False`로 만듭니다(교안 2-6). 트리는 `random_state=42`를 줍니다.

<details><summary>힌트</summary>

```python
logit_1 = make_pipe(LogisticRegression(max_iter=3000))
logit_001 = make_pipe(LogisticRegression(max_iter=3000, C=0.01))
tree_5 = make_pipe(DecisionTreeClassifier(max_depth=5, random_state=42), scale=False)
compare("C=1", logit_1, "C=0.01", logit_001)
```
</details>

In [19]:
logit_1   = make_pipe(LogisticRegression(max_iter=3000))
logit_001 = make_pipe(LogisticRegression(max_iter=3000, C=0.01))
tree_5    = make_pipe(DecisionTreeClassifier(max_depth=5, random_state=42), scale=False)
tree_8    = make_pipe(DecisionTreeClassifier(max_depth=8, random_state=42), scale=False)

compare("C=1", logit_1, "C=0.01", logit_001)        # ①
compare("트리 깊이 5", tree_5, "트리 깊이 8", tree_8)  # ②
compare("C=1", logit_1, "트리 깊이 8", tree_8)        # ③


C=1: [0.765 0.765 0.759 0.769 0.762] | 평균 0.764 ± 0.003
C=0.01: [0.763 0.76  0.759 0.765 0.761] | 평균 0.761 ± 0.002
fold별 차이 (B - A): [-0.002 -0.005 -0.    -0.004 -0.001] | 평균 차이 -0.002

트리 깊이 5: [0.688 0.713 0.7   0.691 0.699] | 평균 0.698 ± 0.009
트리 깊이 8: [0.705 0.715 0.711 0.693 0.711] | 평균 0.707 ± 0.008
fold별 차이 (B - A): [0.016 0.002 0.011 0.002 0.012] | 평균 차이 +0.009

C=1: [0.765 0.765 0.759 0.769 0.762] | 평균 0.764 ± 0.003
트리 깊이 8: [0.705 0.715 0.711 0.693 0.711] | 평균 0.707 ± 0.008
fold별 차이 (B - A): [-0.06  -0.05  -0.048 -0.076 -0.051] | 평균 차이 -0.057



> **[판단]** 비교 ①: fold별 차이의 방향은 일관되나요? 크기는 실무적으로 의미가 있나요? 둘 중 무엇을 고르겠습니까?
>
> 나의 답: 방향은 일관된다. fold별 차이(C=0.01 − C=1)가 -0.002, -0.005, -0.000, -0.004, -0.001로 다섯 번 모두 C=0.01이 같거나 낮다. 하지만 평균 차이가 -0.002로 fold 편차(0.003)보다 작고, 상위 1,000명 가입으로 따져도 몇 명 수준이라 실무적으로는 의미가 없다. 둘 중이면 기본값 C=1을 고르겠다. 조금이라도 높고, 굳이 바꿀 이유가 없다.

> **[판단]** 비교 ②: 다섯 번 모두 같은 쪽이 이겼나요? 차이의 크기는 fold마다 어떻게 다른가요? "깊이 8이 낫다"고 보고하겠습니까?
>
> 나의 답: 다섯 번 모두 깊이 8이 이겼다(+0.016, +0.002, +0.011, +0.002, +0.012). 하지만 차이 크기가 fold마다 0.002~0.016으로 8배까지 들쭉날쭉하고, 평균 차이 +0.009는 각 모델의 fold 편차(0.008~0.009)와 비슷하다. 방향은 일관되지만 크기가 작고 불안정하다. '깊이 8이 약간 나은 경향이 있다' 정도로만 말하고, '낫다'고 단정해 보고하지는 않겠다. 두 트리 모두 로지스틱(0.764)보다 한참 낮으니 실무에서는 이 비교 자체가 중요하지 않다.

> **[판단]** 비교 ③: 차이가 표준편차의 몇 배쯤 되나요? 이 비교는 "낫다"고 보고할 수 있나요? 3-2의 예상은 맞았나요?
>
> 나의 답: 평균 차이 -0.057은 로지스틱의 표준편차(0.003)의 약 19배, 트리의 표준편차(0.008)의 약 7배다. fold별로도 -0.048~-0.076으로 다섯 번 모두 로지스틱이 크게 이겼다. '로지스틱 회귀(C=1)가 결정트리 깊이 8보다 낫다'고 보고할 수 있다. 3-2의 예상(③만 자신 있게 보고 가능)이 맞았다.

## 3-4. 교차검증으로 설정 고르기

**[직접 작성]** `GridSearchCV`로 로지스틱 회귀의 `C`를 0.01, 0.1, 1, 10 중에서 고르세요. 가장 좋은 설정(`best_params_`)과, 후보별 교차검증 평균·표준편차 표를 출력합니다.

<details><summary>힌트</summary>

파이프라인 안의 설정은 `단계이름__설정이름`으로 가리킵니다.
```python
search = GridSearchCV(make_pipe(LogisticRegression(max_iter=3000)), {"clf__C": [0.01, 0.1, 1, 10]},
                      cv=cv, scoring="roc_auc")
search.fit(train_df[features], y_train)
pd.DataFrame(search.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]]
```
</details>

In [20]:
search = GridSearchCV(make_pipe(LogisticRegression(max_iter=3000)),
                      {"clf__C": [0.01, 0.1, 1, 10]},
                      cv=cv, scoring="roc_auc")
search.fit(train_df[features], y_train)

print("best:", search.best_params_)
pd.DataFrame(search.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]].round(4)


best: {'clf__C': 0.1}


,param_clf__C,mean_test_score,std_test_score
0,0.01,0.7615,0.0021
1,0.10,0.7641,0.0029
2,1.00,0.7639,0.0031
3,10.00,0.7638,0.0031


> **[판단]** `GridSearchCV`가 고른 설정과 다른 후보들의 차이는 3-3의 기준으로 구별할 만한 크기인가요? 여러분이라면 어떤 `C`를 쓰겠습니까?
>
> 나의 답: 구별할 수 없다. GridSearchCV가 C=0.1(0.7641)을 골랐지만 C=1(0.7639), C=10(0.7638)과의 차이는 0.0002~0.0003으로 표준편차(약 0.003)의 1/10 수준이다. C=0.01(0.7615)만 0.0026 낮은데, 이것도 편차 정도라 3-3의 기준으로는 크다고 할 수 없다. 나는 C=1(기본값)을 쓰겠다. 0.1~10 사이는 사실상 같고, 지금까지 기록한 모델과 설정이 같아 설명하기도 쉽다. 'best_params_가 골랐으니 쓴다'가 아니라 '차이가 없으니 단순한 쪽'으로 고른다.

## 3-5. 시간이 있는 데이터는 시간 순으로

이 파일의 행은 2008년 5월부터 2010년 11월까지 **날짜순**으로 정렬되어 있습니다. **[제공]** 셀은 행을 순서대로 열 묶음으로 나눠 가입률을 보여 줍니다.

In [21]:
# [제공] 시간 순서대로 본 가입률
deciles = pd.Series(np.arange(len(df)) * 10 // len(df), index=df.index)
print(df.groupby(deciles)["target"].mean().round(3).rename("가입률").rename_axis("구간 (0 = 처음 10%)"))

구간 (0 = 처음 10%)
0    0.030
1    0.037
2    0.048
3    0.064
4    0.061
5    0.062
6    0.105
7    0.131
8    0.161
9    0.471
Name: 가입률, dtype: float64


**[직접 작성]** 앞 80%의 행으로 학습하고 뒤 20%의 행으로 평가하는 **시간 순 분할**을 만드세요. 2-4와 같은 파이프라인을 학습용(앞 80%)으로 `fit`하고, 평가용(뒤 20%)의 확률을 `prob_time`에 저장한 뒤 `record("Chapter 3 모델", y_test_time, prob_time, split="시간 순")`으로 기록하세요.

<details><summary>힌트</summary>

```python
cut = int(len(df) * 0.8)
train_time, test_time = df.iloc[:cut], df.iloc[cut:]
y_test_time = test_time["target"]
```
`make_pipe(LogisticRegression(max_iter=3000))`로 새 파이프라인을 만들어 `train_time[features]`로 `fit`합니다.
</details>

In [22]:
cut = int(len(df) * 0.8)
train_time, test_time = df.iloc[:cut], df.iloc[cut:]
y_train_time, y_test_time = train_time["target"], test_time["target"]

pipe_time = make_pipe(LogisticRegression(max_iter=3000))
pipe_time.fit(train_time[features], y_train_time)
prob_time = pipe_time.predict_proba(test_time[features])[:, 1]

record("Chapter 3 모델", y_test_time, prob_time, split="시간 순")


기록: [Chapter 3 모델 / 시간 순] ROC AUC 0.692 | 상위 1,000명 가입 532명 (무작위의 1.7배)


> **[판단]** 시간 순으로 나누면 ROC AUC가 얼마나 달라지나요? 왜 이런 차이가 생길까요? 실제 캠페인과 더 가까운 조건은 어느 쪽인가요?
>
> 나의 답: 무작위 분할 0.773 → 시간 순 분할 0.692로 0.08 떨어졌다. 3-1의 교차검증 편차(0.003)의 수십 배라 우연으로 볼 수 없다. 데이터가 시간에 따라 크게 변했기 때문이다. 처음 10% 구간의 가입률은 3%인데 마지막 10% 구간은 47%다. 시기마다 고객 구성, 경제 상황, 캠페인 방식이 달랐을 것이다. 무작위 분할은 같은 시기의 고객이 학습용과 평가용에 섞여 있어 '이미 본 시기'를 맞히는 시험이 된다. 시간 순 분할은 과거로 배워 미래를 맞혀야 한다. 실제 캠페인은 과거 데이터로 학습해 다음(미래) 고객에게 쓰므로 시간 순 분할이 더 가까운 조건이다.

> **[판단]** 그런데 시간 순 분할의 "상위 1,000명 가입"은 무작위 분할보다 많거나 적은가요? ROC AUC는 낮아졌는데 이 숫자는 왜 이렇게 나왔을까요? 결과표의 "평가 가입률"과 "배수" 열을 함께 보세요.
>
> 나의 답: 더 많다. 무작위 분할 464명 → 시간 순 분할 532명. 하지만 모델이 좋아져서가 아니다. 평가 가입률이 0.117 → 0.316으로 세 배 가까이 높아서, 무작위로 1,000명만 골라도 117명 → 316명이 가입하는 기간이기 때문이다. 배수로 보면 4.0배 → 1.7배로 모델의 '골라내는 힘'은 오히려 크게 줄었다. 상위 1,000명 가입 같은 절대 숫자는 평가 기간의 기본 가입률에 크게 좌우되므로, 배수나 ROC AUC와 함께 봐야 한다.

---
# Part 4. 보고할 숫자 정하기

이제 질문으로 돌아옵니다. **마케팅팀에 "다음 캠페인에서 기대할 가입 수"로 보고할 숫자는 어느 것인가?**

## 4-1. 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 네 가지 조합의 점수를 한 표에 놓습니다.

| | 무작위 분할 | 시간 순 분할 |
| --- | --- | --- |
| 통화 시간 없음 | 1-2 (기록됨) | 3-5 (기록됨) |
| 통화 시간 있음 | 1-3 (기록됨) | 4-2에서 만듦 |

> **[판단]** 예상: 네 조합 중 ROC AUC가 가장 높은 것과 가장 낮은 것은 무엇일까요? 다음 캠페인에서 실제로 나올 성과에 가장 가까운 것은 어느 칸일까요?
>
> 나의 답: 가장 높은 것: 통화 시간 있음 + 무작위 분할(누수와 운 좋은 분할이 겹침). 가장 낮은 것: 통화 시간 없음 + 시간 순 분할. 실제 성과에 가장 가까운 것도 통화 시간 없음 + 시간 순 분할이다. 전화 전에 알 수 있는 변수만 쓰고, 과거로 배워 미래를 평가하는 조건이 실제 캠페인과 같기 때문이다.

## 4-2. 네 조합 비교하기

**[직접 작성]** 남은 한 칸(통화 시간 있음 + 시간 순 분할)을 만드세요. `features + ["duration"]`을 쓰는 파이프라인을 만들어야 하므로, `make_pipe`의 숫자 열 목록에 `duration`을 더해야 합니다. 확률을 기록하고 `show_log()`로 결과표 전체를 보세요.

<details><summary>힌트</summary>

`make_pipe`는 `num_cols`를 씁니다. 이 셀에서만 직접 만들면 됩니다.
```python
pre_dur = ColumnTransformer([
    ("num", StandardScaler(), num_cols + ["duration"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
```
</details>

In [23]:
features_dur = features + ["duration"]
pre_dur = ColumnTransformer([
    ("num", StandardScaler(), num_cols + ["duration"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_dur_time = Pipeline([("pre", pre_dur), ("clf", LogisticRegression(max_iter=3000))])

pipe_dur_time.fit(train_time[features_dur], y_train_time)
prob_dur_time = pipe_dur_time.predict_proba(test_time[features_dur])[:, 1]
record("통화 시간 추가", y_test_time, prob_dur_time, split="시간 순")

show_log()


기록: [통화 시간 추가 / 시간 순] ROC AUC 0.782 | 상위 1,000명 가입 598명 (무작위의 1.9배)


,실험,분할,ROC AUC,PR AUC,평가 가입률,"상위 1,000명 가입","무작위 1,000명 기대",배수
0,Chapter 3 모델,무작위,0.773,0.411,0.117,464,117,4.0
1,Chapter 3 모델,무작위,0.773,0.411,0.117,464,117,4.0
2,통화 시간 추가,무작위,0.906,0.544,0.117,579,117,4.9
3,파이프라인,무작위,0.773,0.411,0.117,465,117,4.0
4,Chapter 3 모델,시간 순,0.692,0.496,0.316,532,316,1.7
5,통화 시간 추가,시간 순,0.782,0.569,0.316,598,316,1.9


> **[판단]** 예상은 맞았나요? 네 조합 중 마케팅팀에 보고할 점수는 어느 칸이고, 왜 그런가요? 통화 시간을 넣은 두 칸은 왜 보고할 수 없나요?
>
> 나의 답: 맞았다. 가장 높은 칸은 통화 시간 + 무작위(0.906), 가장 낮은 칸은 통화 시간 없음 + 시간 순(0.692)이다. 보고할 점수는 '통화 시간 없음 + 시간 순 분할': ROC AUC 0.692, 상위 1,000명 중 532명 가입(무작위의 1.7배). 실제 캠페인 조건(전화 전 정보만, 과거 → 미래)과 같은 조건에서 잰 숫자이기 때문이다. 통화 시간을 넣은 두 칸(0.906, 0.782)은 보고할 수 없다. 통화 시간은 전화가 끝나야 알 수 있어서, 누구에게 전화할지 정하는 시점에는 그 모델을 돌릴 수 없다. 점수가 높아도 실제로 쓸 수 없는 모델의 점수다.

> **[판단]** 마케팅팀은 "1,000통이면 몇 명이 가입하나요?"라고 묻습니다. 시간 순 분할의 "상위 1,000명 가입" 숫자를 그대로 약속해도 될까요? 다음 캠페인의 가입률이 지난 기간과 다를 수 있다는 점을 생각하면, 어떤 형태로 말하는 것이 정직할까요?
>
> 나의 답: 그대로 약속하면 안 된다. 532명은 평가 기간의 가입률이 31.6%였기 때문에 나온 숫자다(무작위로 골라도 316명). 다음 캠페인의 가입률이 지난 기간과 다르면 같은 모델로도 절대 수는 크게 달라진다. 예를 들어 가입률이 10%라면 1.7배로 잡아 약 170명이다. 정직한 표현은 '같은 1,000통을 무작위로 걸 때보다 약 1.7배 많은 가입을 기대할 수 있다'처럼 기준 대비 배수로 말하는 것이다. 절대 수는 '다음 캠페인 가입률이 ○%라면 약 ○명'처럼 가정을 달아 범위로 제시한다. 캠페인 초반 실적을 보고 숫자를 다시 맞추겠다는 조건도 붙인다.

## 4-3. 점검표 채우기

교안 3-6의 여섯 질문을 **보고하려는 모델**(통화 시간 없음, 파이프라인)에 대해 하나씩 답하세요. 각 질문에 "통과 / 미통과 / 확인 못 함"과 근거를 적습니다.

> **[판단]** 점검표:
>
> ① 쓴 변수가 모두 예측 시점에 존재하는가: 확인 못 함 — duration은 뺐다. 하지만 campaign(마지막 통화 포함)과 month·day(마지막 통화 날짜)는 전화 전에 확정되는 값인지 데이터 설명만으로는 확인할 수 없다.
>
> ② 전처리 기준을 학습용으로만 잡았는가: 통과 — ColumnTransformer + Pipeline을 학습용으로만 fit했고, 교차검증에서도 fold마다 학습 부분으로만 다시 fit된다.
>
> ③ 설정을 고를 때 평가용 데이터를 쓰지 않았는가: 통과 — C는 학습용 5겹 교차검증(GridSearchCV)으로 비교했고, 평가용은 마지막 기록에만 썼다.
>
> ④ 비교한 차이가 fold 편차보다 크고 의미 있는가: 통과 — 로지스틱 vs 트리는 차이 0.057로 편차의 7~19배라 로지스틱을 골랐다. C 후보 간 차이(0.0002)는 편차보다 작아 의미 없다고 보고 기본값 C=1을 유지했다.
>
> ⑤ 평가용이 학습용보다 나중인가: 통과 — 보고하는 숫자는 앞 80%로 학습하고 뒤 20%로 평가한 시간 순 분할 결과다(무작위 분할 점수는 보고하지 않음).
>
> ⑥ 베이스라인과 현행 규칙보다 얼마나 나은가: 부분 통과 — 무작위로 거는 것(베이스라인)보다 상위 1,000명 가입이 1.7배(532명 vs 316명)다. 현재 마케팅팀이 쓰는 대상 선정 규칙과는 비교하지 못했다(확인 못 함).
>
> 나의 답: 위 항목별로 작성

## 4-4. 마케팅팀 보고 메모

분석 결과를 마케팅팀장에게 메시지로 보낸다고 생각하고 쓰세요. 팀장은 머신러닝 용어를 모릅니다. 아래 네 가지가 모두 들어가야 합니다.

1. **보고하는 숫자와 그 근거**: 어떤 조건(분할 방식)에서 잰 숫자인지
2. **통화 시간 모델을 쓰지 않는 이유**: 점수가 더 높은데도 쓰지 않는 이유를 쉬운 말로
3. **1,000통의 기대 효과**: 절대 숫자를 약속할 수 있는지, 아니라면 어떤 형태로 말하는지
4. **주의할 점**: 다음 캠페인의 가입률이 달라질 가능성, 아직 확인하지 못한 것

> **[판단]** 마케팅팀 보고 메모:
>
> 1. 보고하는 숫자와 근거: 과거 기간 데이터로 모델을 만들고, 그 뒤 기간 고객에게 적용해 봤습니다. 모델이 고른 상위 1,000명에게 전화하면 532명이 가입해, 무작위로 1,000명에게 걸 때(316명)보다 약 1.7배 많았습니다. 실제 캠페인처럼 '과거로 배워 미래에 쓰는' 조건에서 잰 숫자입니다.
>
> 2. 통화 시간 모델을 쓰지 않는 이유: 통화 시간을 넣으면 점수가 훨씬 높게 나오지만, 통화 시간은 전화를 끝내야 알 수 있습니다. '누구에게 전화할지'를 정할 때는 아직 없는 정보라 실제로는 쓸 수 없습니다. 가입하는 고객일수록 통화가 길어지는 것이라, 결과를 보고 결과를 맞히는 셈입니다.
>
> 3. 1,000통의 기대 효과: '1,000통에 ○명'을 확정해서 약속하기는 어렵습니다. 같은 1,000통을 무작위로 거는 것보다 약 1.7배 가입을 기대할 수 있다고 말씀드리겠습니다. 예를 들어 다음 캠페인 가입률이 10% 수준이면 약 170명, 30% 수준이면 약 500명입니다.
>
> 4. 주의할 점: 지난 기간에는 시기마다 가입률이 3%에서 47%까지 크게 달라졌습니다. 다음 캠페인 가입률에 따라 실제 숫자가 달라질 수 있으니, 초반 실적을 보고 기대치를 다시 맞추겠습니다. 이번 캠페인 통화 횟수 같은 일부 정보는 전화 전에 알 수 있는지 아직 확인하지 못했습니다. 현재 쓰시는 대상 선정 방식과도 아직 비교하지 못했습니다.
>
> 나의 답: 위 항목별로 작성

---
# 마무리

## 이 Chapter의 판단 포인트

> **이 정보는 예측 시점에 알 수 있는가? 그리고 두 모델의 점수 차이는 교차검증 편차보다 큰가?**

## 개인 검증 체크리스트에 추가

오늘 실습에서 내가 실제로 겪은 일을 바탕으로, 체크리스트의 줄을 내 말로 적으세요.

예: "점수가 갑자기 좋아진 변수를 보면, 그 값이 언제 기록되는지부터 확인한다."

> **[판단]** 나의 체크리스트 6번(누수를 점검할 때):
>
> 나의 답: 점수가 갑자기 크게 오르면 기뻐하기 전에 그 변수가 언제 기록되는지(예측 시점에 존재하는지)부터 확인한다. 전처리 기준(스케일러, 인코더, 특히 타깃이 들어가는 인코딩)은 학습용으로만 잡고, 파이프라인으로 묶어 순서를 틀릴 여지를 없앤다. 나누기 전 전체 데이터에 get_dummies·fit을 하고 있지 않은지 본다.

> **[판단]** 나의 체크리스트 7번(점수를 비교하고 보고할 때):
>
> 나의 답: 두 모델의 차이는 평균만 보지 않고 같은 fold에서의 차이와 방향, 표준편차 대비 크기로 판단한다. 차이가 편차 수준이면 단순한 쪽을 고른다. 시간이 있는 데이터는 시간 순 분할 점수를 보고하고, 상위 N명 같은 절대 숫자는 평가 기간의 기본 비율에 좌우되므로 배수와 가정을 함께 적는다.

## 제출물

- 이 노트북 (모든 **[판단]** 칸을 채운 상태)

수고했습니다. 다음 Chapter 5에서는 트리를 여러 개 모아 더 강한 모델을 만들고, 그 모델이 무엇을 보고 예측하는지 SHAP으로 설명합니다.